# IMG_7916 → walkable splat (splatfacto)

This is **not** the old `model.splat` notebook. That loop kept writing the same 1 MB cloud.

This notebook trains **Nerfstudio splatfacto**, the standard phone-video Gaussian splat. Runtime → **GPU** first.

Put `IMG_7916.mp4` (the 379 MB copy of the same 90-second walk) in Google Drive at `MyDrive/IMG_7916.mp4`. Do not upload the 660 MB MOV. The trainer keeps about 180 frames and shrinks them, so the extra 4K file only makes the upload more likely to fail.

A finished room is a **new** file, tens of megabytes, named `img7916_splatfacto.ply`. The old `point_cloud.ply` at 1 MB is the September cloud. Do not open that one.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU, then rerun this cell"
print("GPU", torch.cuda.get_device_name(0), torch.__version__)

!apt-get -qq update && apt-get -qq install -y colmap ffmpeg
!pip -q install nerfstudio
import nerfstudio
print("splatfacto ready", nerfstudio.__file__)

In [ ]:
from google.colab import drive
import os, glob, shutil, subprocess

os.environ["QT_QPA_PLATFORM"] = "offscreen"
drive.mount("/content/drive")
video = "/content/drive/MyDrive/IMG_7916.mp4"
assert os.path.isfile(video), video
print("video bytes", os.path.getsize(video))

def run(cmd):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

if os.path.isdir("data/img7916"):
    shutil.rmtree("data/img7916")
run(["ns-process-data", "video", "--data", video, "--output-dir", "data/img7916", "--num-frames-target", "180", "--matching-method", "sequential", "--no-gpu"])
run(["ns-train", "splatfacto", "--data", "data/img7916", "--output-dir", "outputs", "--max-num-iterations", "30000", "--viewer.quit-on-train-completion", "True"])

configs = sorted(glob.glob("outputs/**/config.yml", recursive=True))
assert configs, "training did not write a config"
config = configs[-1]
print("config", config)
run(["ns-export", "gaussian-splat", "--load-config", config, "--output-dir", "exports/img7916"])

plys = glob.glob("exports/img7916/**/*.ply", recursive=True) + glob.glob("exports/img7916/*.ply")
assert plys, "export wrote no ply"
ply = max(plys, key=os.path.getsize)
size = os.path.getsize(ply)
print("exported", ply, size)
if size < 5_000_000:
    raise SystemExit("Export is under 5 MB. That is the sparse cloud again. Do not replace the viewer with it.")

dest = "/content/drive/MyDrive/indoor_splat/img7916_splatfacto.ply"
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copy2(ply, dest)
print("Drive", dest, os.path.getsize(dest))
